In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import joblib
# load the model
filename='randomforest.pkl'
rf_model = joblib.load(open(filename, 'rb'))

In [3]:
def supply_newdata(Tenure ,
        PreferredLoginDevice ,
        CityTier,
        WarehouseToHome      ,
        PreferredPaymentMode ,
        Gender  ,
        HourSpendOnApp       ,
        NumberOfDeviceRegistered       ,
        PreferedOrderCat     ,
        SatisfactionScore    ,
        MaritalStatus        ,
        NumberOfAddress      ,
        Complain,
        OrderAmountHikeFromlastYear    ,
        CouponUsed ,
        OrderCount ,
        DaySinceLastOrder    ,
        CashbackAmount):
    e_commerce = {"Tenure":Tenure ,
        "PreferredLoginDevice":PreferredLoginDevice ,
        "CityTier":CityTier,
        "WarehouseToHome":WarehouseToHome      ,
        "PreferredPaymentMode":PreferredPaymentMode ,
        "Gender" : Gender ,
        "HourSpendOnApp" :HourSpendOnApp      ,
        "NumberOfDeviceRegistered" :NumberOfDeviceRegistered      ,
        "PreferedOrderCat" :PreferedOrderCat    ,
        "SatisfactionScore" :SatisfactionScore   ,
        "MaritalStatus" :MaritalStatus       ,
        "NumberOfAddress" :NumberOfAddress     ,
        "Complain":Complain,
        "OrderAmountHikeFromlastYear" :OrderAmountHikeFromlastYear   ,
        "CouponUsed":CouponUsed ,
        "OrderCount":OrderCount ,
        "DaySinceLastOrder" :DaySinceLastOrder   ,
        "CashbackAmount":CashbackAmount

    }
    df=pd.DataFrame(e_commerce)

    return df

def clean_data(e_commerce):
    categorical_columns = [
    'PreferredLoginDevice',
    'PreferredPaymentMode',
    'Gender',
    'PreferedOrderCat',
    'MaritalStatus'
        ]

    e_commerce = pd.get_dummies(
        e_commerce,
        columns=categorical_columns,
        drop_first=True,
        dtype=int
    )


    # Orderd placed per month of tenure
    e_commerce['OrderFrequency']= e_commerce['OrderCount'] /(e_commerce['Tenure']+1)

    # Share of orders that used coupon
    e_commerce['CouponUsageRate'] =e_commerce['CouponUsed'] / (e_commerce['OrderCount'] + 1)

    # Days since last order relative to tenure — flags customers who have gone quiet relative to how long they have been a customer, not just in absolute days
    e_commerce['RecencyRatio']=e_commerce['DaySinceLastOrder'] /(e_commerce['Tenure'] + 1)

    # Engagement score (app usage with no. registered devices) 
    e_commerce['EngagementScore'] = (
        e_commerce['HourSpendOnApp'] * e_commerce['NumberOfDeviceRegistered'])

    return e_commerce


def new_answer(ecommerce):
    return rf_model.predict(ecommerce)


Adding an llm

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]

from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile" 

In [5]:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
             temperature=1, max_tokens=500):
    response = client.chat.completions.create(
         model=MODEL,
         messages=[
             {"role": "system", "content": system_prompt},
             {"role": "user",   "content": user_prompt},
         ],
         temperature=temperature,
         max_tokens=max_tokens,
     )
    return response.choices[0].message.content



In [6]:
EXLANAION_PROMPT={"sys_p": """ you are a business advisor for an ecommerce chain""",
                  "user_p": """Take a panda dataframe of a customer and their churn result (1 leave, 0 means stay)and output 
                   1. reasons why they might leave if value is 1 and reasons they might stay
                    2. Possible steps to make the customer stay
                    3. Business advice to ensurecustomers churn value remains at 0-meaning thst they do not leave
                    ensure that it is clear and straightforward """}




main part/class

In [7]:
#ecommerce=supply_data()
#new_ecommerce=clean_data(ecommerce)
#n_ans=new_answer(new_ecommerce)
#a= ask_llm(user_prompt=f'{EXLANAION_PROMPT["user_p"]} panda dataframe= {ecommerce}, churn result{n_ans}', system_prompt=EXLANAION_PROMPT["sys_p"])